In [ ]:
# @colab-setup
# Run this cell first. On Colab it installs the deps; locally it is a no-op.
import sys
if "google.colab" in sys.modules:
    %pip install -q pyworms requests pandas

# 03 — WoRMS REST API + pyworms

**Time budget:** ~15 min · **Goal:** turn the raw `scientificName` strings we pulled from OBIS in Notebooks 01–02 into authoritative taxonomy — stable AphiaIDs, accepted names, full classification trees.

The [World Register of Marine Species](https://www.marinespecies.org/) (WoRMS) is the taxonomic backbone OBIS itself uses for name interpretation. Knowing how to query it directly matters because:

- OBIS records carry `scientificName` *as supplied by the publisher* — often misspelt, synonymised, or out of date.
- The `interpreted.*` struct in the Parquet exports already does WoRMS resolution, but if you're working with non-OBIS data (your own collection, GBIF, a fresh DwC archive) you have to do it yourself.
- WoRMS exposes a JSON REST service at `https://www.marinespecies.org/rest/` and a thin Python wrapper, [`pyworms`](https://github.com/iobis/pyworms), maintained by the OBIS Secretariat.

We'll use both — `pyworms` for the common cases, raw `requests` when we need to control batching or fields.

In [ ]:
import pyworms
import requests
import pandas as pd

WORMS = 'https://www.marinespecies.org/rest'

## 1. Single-name lookup

Atlantic cod — one of the top species in the Maritimes Spring RV dataset from Notebook 02. The two-step pattern is: name → AphiaID → full record.

In [ ]:
aphia_id = pyworms.aphiaIDByName('Gadus morhua')
print('AphiaID:', aphia_id)

record = pyworms.aphiaRecordByAphiaID(aphia_id)
{k: record[k] for k in ('AphiaID', 'scientificname', 'authority', 'rank', 'status', 'valid_AphiaID', 'valid_name', 'kingdom', 'phylum', 'class', 'order', 'family')}

## 2. Synonym → accepted name

A name that's still in circulation but has been superseded. WoRMS returns the record for the synonym (`status: "unaccepted"`) with a pointer to the accepted `valid_AphiaID` — always follow that pointer before using the name downstream.

In [ ]:
# 'Clupea harengus harengus' is an obsolete subspecies designation for Atlantic herring.
rec = pyworms.aphiaRecordByAphiaID(pyworms.aphiaIDByName('Clupea harengus harengus'))
print('Looked up :', rec['scientificname'], '— status:', rec['status'])
print('Accepted  :', rec['valid_name'], '(AphiaID', rec['valid_AphiaID'], ')')

# Always resolve to the accepted record:
accepted = pyworms.aphiaRecordByAphiaID(rec['valid_AphiaID'])
accepted['scientificname'], accepted['rank']

## 3. Batch name matching

The everyday use case: you have a column of `scientificName` strings from OBIS, GBIF, or a CSV, and you want an AphiaID for each. `aphiaRecordsByMatchNames` accepts up to 50 names per call and fuzzy-matches them against WoRMS — returns a list-of-lists (multiple candidates per name when the match is ambiguous).

In [ ]:
# A mix taken from the top species in Notebook 02, with one typo to exercise fuzzy match.
names = [
    'Melanogrammus aeglefinus',
    'Gadus morhua',
    'Squalus acanthias',
    'Homarus americanus',
    'Gadus morrhua',  # historical misspelling
]

matches = pyworms.aphiaRecordsByMatchNames(names)

rows = []
for query, candidates in zip(names, matches):
    if not candidates:
        rows.append({'query': query, 'matched_name': None, 'AphiaID': None, 'match_type': None, 'status': None})
        continue
    best = candidates[0]
    rows.append({
        'query':       query,
        'matched_name': best['scientificname'],
        'AphiaID':      best['AphiaID'],
        'match_type':   best.get('match_type'),
        'status':       best['status'],
        'valid_name':   best.get('valid_name'),
    })

pd.DataFrame(rows)

## 4. Full classification tree

The flat `kingdom`/`phylum`/`class`/... fields on a record are convenient but lossy — they only show the seven “standard” ranks. `aphiaClassificationByAphiaID` returns the complete nested hierarchy as WoRMS holds it, including unranked clades.

In [ ]:
cod_aphia = pyworms.aphiaIDByName('Gadus morhua')
tree = pyworms.aphiaClassificationByAphiaID(cod_aphia)

# Walk the nested 'child' chain and flatten to a list of (rank, name).
lineage = []
node = tree
while node:
    lineage.append((node['rank'], node['scientificname']))
    node = node.get('child')

pd.DataFrame(lineage, columns=['rank', 'scientificname'])

## 5. Same call, raw REST

`pyworms` is a thin wrapper — every call maps 1:1 to a URL. Useful to know when you need a field `pyworms` doesn't surface, want to control timeouts, or are debugging from `curl`. All endpoints are documented at <https://www.marinespecies.org/rest/>.

In [ ]:
r = requests.get(f'{WORMS}/AphiaRecordByAphiaID/{cod_aphia}', timeout=30)
r.raise_for_status()
{k: r.json()[k] for k in ('AphiaID', 'scientificname', 'rank', 'status', 'lsid', 'url')}

## 6. Crosswalk: top OBIS species → WoRMS records

The end-to-end shape: pull a species list straight from OBIS (REST this time — same call you'd build in Notebook 01), batch-resolve the names through WoRMS, and emit a clean (scientificName, AphiaID, accepted name) table.

In production you'd skip this for OBIS data — `interpreted.scientificNameID` already carries the AphiaID as a `urn:lsid:marinespecies.org:taxname:NNN` URN. The crosswalk matters when the input is *not* OBIS-interpreted: your own field data, a fresh Darwin Core archive, a GBIF download, or any CSV with a `scientificName` column.

In [ ]:
DEMO_UUID = 'd895e645-a98d-4720-b6fb-332929190f36'  # Maritimes Spring RV Surveys

# Top species via /v3/checklist (same dataset as Notebooks 01–02).
r = requests.get(
    'https://api.obis.org/v3/checklist',
    params={'datasetid': DEMO_UUID, 'size': 10},
    timeout=60,
)
r.raise_for_status()
top = pd.DataFrame(r.json()['results'])[['scientificName', 'records']].head(10)
top

In [ ]:
matches = pyworms.aphiaRecordsByMatchNames(top['scientificName'].tolist())

def pick(candidates):
    if not candidates:
        return {'AphiaID': None, 'accepted_name': None, 'rank': None}
    best = candidates[0]
    return {
        'AphiaID':       best.get('valid_AphiaID') or best['AphiaID'],
        'accepted_name': best.get('valid_name')    or best['scientificname'],
        'rank':          best.get('rank'),
    }

resolved = pd.concat([top.reset_index(drop=True), pd.DataFrame(pick(c) for c in matches)], axis=1)
resolved

## Gotchas to flag in the talk

- **Rate limits**: WoRMS' public REST endpoint is shared infrastructure. Batch with `aphiaRecordsByMatchNames` (≤ 50 names/call) before you parallelise; don't fire one request per row.
- **Marine-only filter**: `aphiaRecordsByMatchNames(..., marine_only=True)` is the default — fine for OBIS, wrong if your dataset includes freshwater or terrestrial species (e.g. river-mouth surveys, GBIF mixed downloads).
- **Always follow `valid_AphiaID`**: a record with `status != 'accepted'` is a synonym — don't store its `AphiaID` as your stable key.
- **For OBIS data, prefer `interpreted.scientificNameID`**: the AphiaID is already in the Parquet exports as an LSID; only do the WoRMS round-trip when the input isn't OBIS-interpreted.

## Wrap-up

Three APIs, three jobs:

1. **OBIS REST** (Notebook 01) — fine-grained queries, faceting, fallback for embargoed data.
2. **OBIS Parquet on S3** (Notebook 02) — bulk + analytical, the production harvester's path.
3. **WoRMS / pyworms** (this notebook) — the authoritative taxonomic backbone behind both.